# 🔐 Senhas em Risco: Big Data e Segurança Digital

---

**Disciplina:** Tópicos de Big Data em Python  
**Instituição:** Unimetrocamp Wyden — Campinas/SP  
**Grupo:**
- Eduardo Gombrade — Análise e Desenvolvimento
- Leandro Schiavo — Documentação
- João Vendito — Dashboard e Visualizações

---

## 📓 Notebook 03 — Análise Exploratória dos Dados (EDA)

**Objetivo deste notebook:**  
Realizar a Análise Exploratória de Dados (EDA — Exploratory Data Analysis) sobre
os datasets limpos, extraindo estatísticas descritivas, padrões e insights
relevantes sobre segurança de senhas digitais.

**Análises executadas:**
1. Estatísticas descritivas gerais
2. Distribuição de força das senhas
3. Análise de comprimento das senhas
4. Composição das senhas (tipos de caracteres)
5. Análise das senhas mais comuns (RockYou)
6. Tempo estimado de quebra por força bruta
7. Análise temporal de vazamentos globais
8. Análise por setor e método de ataque
9. Geração de insights e conclusões


---
## ⚙️ CÉLULA 1 — Importação e Carregamento


In [ ]:
# ============================================================
# IMPORTAÇÃO DAS BIBLIOTECAS E CARREGAMENTO DOS DADOS
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# --- Configurações visuais globais ---
plt.rcParams.update({
    'figure.dpi'       : 120,
    'figure.facecolor' : '#0f0f1a',
    'axes.facecolor'   : '#1a1a2e',
    'axes.edgecolor'   : '#444466',
    'axes.labelcolor'  : '#e0e0ff',
    'axes.titlecolor'  : '#ffffff',
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
    'xtick.color'      : '#aaaacc',
    'ytick.color'      : '#aaaacc',
    'text.color'       : '#e0e0ff',
    'grid.color'       : '#2a2a4a',
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.5,
    'font.family'      : 'DejaVu Sans'
})

PALETA_FORCA    = {'Fraca': '#ff4444', 'Média': '#ffaa00', 'Forte': '#44ff88'}
PALETA_TEMPO    = ['#ff0055','#ff4400','#ff8800','#ffcc00','#aaff00','#00ff88','#00ccff','#0088ff','#8800ff','#ff00cc']
COR_DESTAQUE    = '#7b68ee'
COR_SECUNDARIA  = '#20b2aa'

# --- Carregamento dos dados tratados ---
df_passwords = pd.read_parquet('data/processed/passwords_clean.parquet')
df_rockyou   = pd.read_parquet('data/processed/rockyou_clean.parquet')
df_breaches  = pd.read_parquet('data/processed/breaches_clean.parquet')
df_nordpass  = pd.read_parquet('data/processed/nordpass_clean.parquet')

os.makedirs('outputs/graficos', exist_ok=True)

print('✅ Dados carregados e ambiente configurado!')
print(f'   passwords : {len(df_passwords):,} linhas')
print(f'   rockyou   : {len(df_rockyou):,} linhas')
print(f'   breaches  : {len(df_breaches):,} linhas')
print(f'   nordpass  : {len(df_nordpass):,} linhas')

---
## 📊 CÉLULA 2 — Estatísticas Descritivas Gerais

Calculamos as principais métricas estatísticas do dataset de senhas:
média, mediana, desvio padrão, mínimo e máximo de comprimento.


In [ ]:
# ============================================================
# ESTATÍSTICAS DESCRITIVAS — DATASET PASSWORD STRENGTH
# ============================================================

print('=' * 60)
print('  📋 ESTATÍSTICAS DESCRITIVAS — SENHAS')
print('=' * 60)

total = len(df_passwords)
stats_comp = df_passwords['comprimento'].describe()

print(f'\n  Total de senhas analisadas : {total:>12,}')
print(f'  Comprimento mínimo         : {int(stats_comp["min"]):>12,} caracteres')
print(f'  Comprimento máximo         : {int(stats_comp["max"]):>12,} caracteres')
print(f'  Comprimento médio          : {stats_comp["mean"]:>12.2f} caracteres')
print(f'  Mediana do comprimento     : {stats_comp["50%"]:>12.1f} caracteres')
print(f'  Desvio padrão              : {stats_comp["std"]:>12.2f}')

print('\n  Distribuição por força:')
for label in ['Fraca', 'Média', 'Forte']:
    qtd = (df_passwords['forca_label'] == label).sum()
    pct = qtd / total * 100
    barra = '█' * int(pct / 2)
    print(f'  {label:<6} : {qtd:>9,} ({pct:5.1f}%) {barra}')

print('\n  Uso de tipos de caracteres:')
for col, nome in [('tem_maiuscula','Maiúsculas'), ('tem_minuscula','Minúsculas'),
                  ('tem_numero','Números'), ('tem_simbolo','Símbolos')]:
    qtd = df_passwords[col].sum()
    pct = qtd / total * 100
    print(f'  {nome:<12} : {qtd:>9,} ({pct:5.1f}%)')

print('=' * 60)

---
## 📊 CÉLULA 3 — Gráfico 1: Distribuição de Força das Senhas


In [ ]:
# ============================================================
# GRÁFICO 1 — DISTRIBUIÇÃO DE FORÇA DAS SENHAS
# ============================================================

contagem = df_passwords['forca_label'].value_counts().reindex(['Fraca', 'Média', 'Forte'])
cores    = [PALETA_FORCA[k] for k in contagem.index]
total    = contagem.sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0f0f1a')
fig.suptitle('Distribuição de Força das Senhas', fontsize=15, color='white', fontweight='bold', y=1.01)

# --- Gráfico de barras ---
ax1 = axes[0]
barras = ax1.bar(contagem.index, contagem.values, color=cores, edgecolor='#333355', linewidth=0.8, width=0.55)
ax1.set_title('Contagem por Nível de Força', color='white')
ax1.set_ylabel('Quantidade de Senhas', color='#aaaacc')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{int(x):,}'))
for barra, val in zip(barras, contagem.values):
    pct = val / total * 100
    ax1.text(barra.get_x() + barra.get_width()/2, val + total*0.005,
             f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', color='white', fontsize=9, fontweight='bold')

# --- Gráfico de pizza ---
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    contagem.values,
    labels=contagem.index,
    colors=cores,
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops={'edgecolor': '#0f0f1a', 'linewidth': 2}
)
for text in texts:
    text.set_color('white')
    text.set_fontsize(11)
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
ax2.set_title('Proporção por Nível de Força', color='white')

plt.tight_layout()
plt.savefig('outputs/graficos/01_distribuicao_forca.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print('✅ Gráfico 1 salvo em outputs/graficos/01_distribuicao_forca.png')

---
## 📊 CÉLULA 4 — Gráfico 2: Distribuição do Comprimento das Senhas


In [ ]:
# ============================================================
# GRÁFICO 2 — DISTRIBUIÇÃO DO COMPRIMENTO DAS SENHAS
# ============================================================

# Limitamos a 30 caracteres para melhor visualização (outliers extremos são raros)
df_plot = df_passwords[df_passwords['comprimento'] <= 30]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')
fig.suptitle('Análise de Comprimento das Senhas', fontsize=15, color='white', fontweight='bold')

# --- Histograma geral ---
ax1 = axes[0]
ax1.hist(df_plot['comprimento'], bins=29, color=COR_DESTAQUE, edgecolor='#0f0f1a', alpha=0.85)
ax1.axvline(df_plot['comprimento'].mean(),   color='#ffcc00', linestyle='--', linewidth=1.5, label=f'Média: {df_plot["comprimento"].mean():.1f}')
ax1.axvline(df_plot['comprimento'].median(), color='#00ffcc', linestyle='--', linewidth=1.5, label=f'Mediana: {df_plot["comprimento"].median():.0f}')
ax1.axvline(8, color='#ff4444', linestyle=':', linewidth=1.5, label='Mín. recomendado: 8')
ax1.set_title('Histograma de Comprimento', color='white')
ax1.set_xlabel('Comprimento (caracteres)')
ax1.set_ylabel('Frequência')
ax1.legend(facecolor='#1a1a2e', labelcolor='white', fontsize=9)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# --- Barras empilhadas por força vs categoria de comprimento ---
ax2 = axes[1]
ordem_cat = ['Muito Curta (≤6)', 'Curta (7-8)', 'Média (9-12)', 'Longa (13-16)', 'Muito Longa (>16)']
pivot = df_passwords.groupby(['categoria_comprimento', 'forca_label']).size().unstack(fill_value=0)
pivot = pivot.reindex(ordem_cat).fillna(0)
# Garante que todas as colunas de força existam
for col in ['Fraca', 'Média', 'Forte']:
    if col not in pivot.columns:
        pivot[col] = 0
pivot = pivot[['Fraca', 'Média', 'Forte']]

pivot.plot(
    kind='bar',
    stacked=True,
    ax=ax2,
    color=[PALETA_FORCA['Fraca'], PALETA_FORCA['Média'], PALETA_FORCA['Forte']],
    edgecolor='#0f0f1a',
    linewidth=0.5
)
ax2.set_title('Força por Categoria de Comprimento', color='white')
ax2.set_xlabel('Categoria de Comprimento')
ax2.set_ylabel('Quantidade')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=25, ha='right', fontsize=8)
ax2.legend(title='Força', facecolor='#1a1a2e', labelcolor='white', title_fontsize=9)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('outputs/graficos/02_comprimento_senhas.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print('✅ Gráfico 2 salvo em outputs/graficos/02_comprimento_senhas.png')

---
## 📊 CÉLULA 5 — Gráfico 3: Composição das Senhas


In [ ]:
# ============================================================
# GRÁFICO 3 — COMPOSIÇÃO DAS SENHAS (TIPOS DE CARACTERES)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')
fig.suptitle('Composição das Senhas por Tipo de Caractere', fontsize=15, color='white', fontweight='bold')

total = len(df_passwords)
tipos = {
    'Minúsculas' : df_passwords['tem_minuscula'].sum(),
    'Números'    : df_passwords['tem_numero'].sum(),
    'Maiúsculas' : df_passwords['tem_maiuscula'].sum(),
    'Símbolos'   : df_passwords['tem_simbolo'].sum()
}
cores_tipos = ['#4488ff', '#ffaa00', '#ff6688', '#44ffaa']

# --- Barras horizontais de uso ---
ax1 = axes[0]
nomes  = list(tipos.keys())
valores = [v / total * 100 for v in tipos.values()]
barras = ax1.barh(nomes, valores, color=cores_tipos, edgecolor='#0f0f1a', height=0.5)
for barra, val, qtd in zip(barras, valores, tipos.values()):
    ax1.text(val + 0.5, barra.get_y() + barra.get_height()/2,
             f'{val:.1f}%  ({qtd:,})', va='center', color='white', fontsize=9)
ax1.set_xlim(0, 115)
ax1.set_title('% de Senhas que Contêm Cada Tipo', color='white')
ax1.set_xlabel('Percentual (%)')
ax1.axvline(100, color='#444466', linestyle='--', alpha=0.5)

# --- Distribuição de quantos tipos são usados por senha ---
ax2 = axes[1]
cont_tipos = df_passwords['tipos_usados'].value_counts().sort_index()
cores_qtd  = ['#ff4444', '#ff8800', '#ffcc00', '#44ff88', '#00ccff']
barras2 = ax2.bar(
    [f'{i} tipo(s)' for i in cont_tipos.index],
    cont_tipos.values,
    color=cores_qtd[:len(cont_tipos)],
    edgecolor='#0f0f1a'
)
for barra, val in zip(barras2, cont_tipos.values):
    pct = val / total * 100
    ax2.text(barra.get_x() + barra.get_width()/2, val + total*0.003,
             f'{pct:.1f}%', ha='center', color='white', fontsize=9, fontweight='bold')
ax2.set_title('Quantidade de Tipos por Senha', color='white')
ax2.set_xlabel('Tipos de caracteres utilizados')
ax2.set_ylabel('Quantidade de Senhas')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('outputs/graficos/03_composicao_senhas.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print('✅ Gráfico 3 salvo em outputs/graficos/03_composicao_senhas.png')

---
## 📊 CÉLULA 6 — Gráfico 4: Top 20 Senhas Mais Usadas (RockYou)


In [ ]:
# ============================================================
# GRÁFICO 4 — TOP 20 SENHAS MAIS USADAS (ROCKYOU)
# ============================================================

top20 = df_rockyou.head(20).copy()

fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor('#0f0f1a')

# Degradê de cores do vermelho para o amarelo
n = len(top20)
cores = [plt.cm.YlOrRd(0.4 + 0.6 * (1 - i / n)) for i in range(n)]

barras = ax.barh(
    top20['senha'][::-1],
    top20['frequencia'][::-1],
    color=cores[::-1],
    edgecolor='#0f0f1a',
    height=0.7
)

# Rótulos nas barras
for barra, freq, pct in zip(barras, top20['frequencia'][::-1], top20['pct_uso'][::-1]):
    ax.text(barra.get_width() + max(top20['frequencia']) * 0.01,
            barra.get_y() + barra.get_height() / 2,
            f'{freq:,}  ({pct:.2f}%)', va='center', color='#ccccee', fontsize=8.5)

ax.set_title('Top 20 Senhas Mais Usadas — Vazamento RockYou (2009)', color='white', fontsize=13, fontweight='bold')
ax.set_xlabel('Frequência de Uso', color='#aaaacc')
ax.set_xlim(0, max(top20['frequencia']) * 1.25)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Anotação explicativa
ax.text(0.98, 0.02,
        'Fonte: RockYou Data Breach (2009)\n14,3 milhões de senhas vazadas',
        transform=ax.transAxes, ha='right', va='bottom',
        color='#888899', fontsize=8, style='italic')

plt.tight_layout()
plt.savefig('outputs/graficos/04_top20_rockyou.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print('✅ Gráfico 4 salvo em outputs/graficos/04_top20_rockyou.png')

---
## 📊 CÉLULA 7 — Gráfico 5: Tempo Estimado de Quebra por Força Bruta


In [ ]:
# ============================================================
# GRÁFICO 5 — TEMPO ESTIMADO DE QUEBRA POR FORÇA BRUTA
# ============================================================

ordem_tempo = [
    'Instantâneo', 'Segundos', 'Minutos', 'Horas',
    'Dias', 'Semanas', 'Meses', 'Anos', 'Décadas', 'Séculos'
]

cont_tempo = df_passwords['tempo_quebra'].value_counts()
cont_tempo = cont_tempo.reindex([t for t in ordem_tempo if t in cont_tempo.index]).dropna()
total = cont_tempo.sum()

fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('#0f0f1a')

cores_tempo = [PALETA_TEMPO[i] for i in range(len(cont_tempo))]

barras = ax.bar(
    cont_tempo.index,
    cont_tempo.values,
    color=cores_tempo,
    edgecolor='#0f0f1a',
    width=0.65
)

for barra, val in zip(barras, cont_tempo.values):
    pct = val / total * 100
    ax.text(barra.get_x() + barra.get_width() / 2, val + total * 0.003,
            f'{pct:.1f}%', ha='center', va='bottom', color='white', fontsize=9, fontweight='bold')

ax.set_title('Distribuição do Tempo Estimado de Quebra por Força Bruta\n(baseado na Hive Systems Password Table 2024)',
             color='white', fontsize=12, fontweight='bold')
ax.set_xlabel('Tempo Estimado para Quebrar a Senha', color='#aaaacc')
ax.set_ylabel('Quantidade de Senhas', color='#aaaacc')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('outputs/graficos/05_tempo_quebra.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print('✅ Gráfico 5 salvo em outputs/graficos/05_tempo_quebra.png')

---
## 📊 CÉLULA 8 — Gráfico 6: Timeline de Vazamentos Globais


In [ ]:
# ============================================================
# GRÁFICO 6 — TIMELINE DE VAZAMENTOS GLOBAIS (2004–2024)
# ============================================================

# Verificamos quais colunas existem no dataset de breaches
print('Colunas disponíveis:', list(df_breaches.columns))

# Agrupamos por ano: quantidade de incidentes e total de registros afetados
col_registros = 'registros_afetados' if 'registros_afetados' in df_breaches.columns else 'records'
col_ano       = 'ano' if 'ano' in df_breaches.columns else 'year'

por_ano = df_breaches.groupby(col_ano).agg(
    incidentes=(col_ano, 'count'),
    total_afetados=(col_registros, 'sum')
).reset_index()
por_ano.columns = ['ano', 'incidentes', 'total_afetados']
por_ano = por_ano.sort_values('ano')

fig, ax1 = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')

# Barras: número de incidentes
ax1.bar(por_ano['ano'].astype(str), por_ano['incidentes'],
        color=COR_DESTAQUE, alpha=0.7, label='Nº de Incidentes', edgecolor='#0f0f1a')
ax1.set_ylabel('Número de Incidentes', color=COR_DESTAQUE)
ax1.tick_params(axis='y', labelcolor=COR_DESTAQUE)
ax1.set_xlabel('Ano')

# Linha: total de registros afetados
ax2 = ax1.twinx()
ax2.plot(por_ano['ano'].astype(str), por_ano['total_afetados'] / 1e6,
         color='#ff4466', linewidth=2.5, marker='o', markersize=5,
         label='Total Afetados (M)', zorder=5)
ax2.set_ylabel('Registros Afetados (Milhões)', color='#ff4466')
ax2.tick_params(axis='y', labelcolor='#ff4466')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}M'))

ax1.set_title('Timeline de Grandes Vazamentos de Dados (2004–2024)',
              color='white', fontsize=13, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

# Legenda combinada
linhas1, labels1 = ax1.get_legend_handles_labels()
linhas2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(linhas1 + linhas2, labels1 + labels2,
           facecolor='#1a1a2e', labelcolor='white', loc='upper left')

plt.tight_layout()
plt.savefig('outputs/graficos/06_timeline_vazamentos.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print('✅ Gráfico 6 salvo em outputs/graficos/06_timeline_vazamentos.png')

---
## 📊 CÉLULA 9 — Gráfico 7: Heatmap — Setor vs Método de Ataque


In [ ]:
# ============================================================
# GRÁFICO 7 — HEATMAP: SETOR vs MÉTODO DE ATAQUE
# ============================================================

# Identificamos as colunas corretas no dataset
col_setor  = 'setor'         if 'setor'         in df_breaches.columns else 'organization_type'
col_metodo = 'metodo_ataque' if 'metodo_ataque' in df_breaches.columns else 'method'

# Filtramos apenas setores e métodos com dados suficientes
top_setores = df_breaches[col_setor].value_counts().head(8).index
top_metodos = df_breaches[col_metodo].value_counts().head(7).index

df_hm = df_breaches[
    df_breaches[col_setor].isin(top_setores) &
    df_breaches[col_metodo].isin(top_metodos)
]

pivot_hm = df_hm.groupby([col_setor, col_metodo]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('#0f0f1a')

sns.heatmap(
    pivot_hm,
    ax=ax,
    cmap='YlOrRd',
    annot=True,
    fmt='d',
    linewidths=0.5,
    linecolor='#0f0f1a',
    cbar_kws={'label': 'Nº de Incidentes'},
    annot_kws={'size': 9, 'color': 'white'}
)

ax.set_title('Heatmap: Setor × Método de Ataque', color='white', fontsize=13, fontweight='bold')
ax.set_xlabel('Método de Ataque', color='#aaaacc')
ax.set_ylabel('Setor', color='#aaaacc')
ax.tick_params(colors='#ccccee')
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)

plt.tight_layout()
plt.savefig('outputs/graficos/07_heatmap_setor_metodo.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print('✅ Gráfico 7 salvo em outputs/graficos/07_heatmap_setor_metodo.png')

---
## 📊 CÉLULA 10 — Gráfico 8: Senhas Mais Comuns — NordPass 2024


In [ ]:
# ============================================================
# GRÁFICO 8 — TOP 15 SENHAS MAIS COMUNS (NORDPASS 2024)
# ============================================================

top15 = df_nordpass.head(15).copy()

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#0f0f1a')

cores = [plt.cm.plasma(0.15 + 0.7 * i / len(top15)) for i in range(len(top15))]

barras = ax.barh(
    top15['senha'][::-1],
    top15['usuarios_afetados'][::-1] / 1e6,
    color=cores[::-1],
    edgecolor='#0f0f1a',
    height=0.65
)

for barra, senha, tempo in zip(barras, top15['senha'][::-1], top15['tempo_para_quebrar'][::-1]):
    ax.text(barra.get_width() + 0.02,
            barra.get_y() + barra.get_height() / 2,
            f'⏱ {tempo}', va='center', color='#ffcc44', fontsize=8)

ax.set_title('Top 15 Senhas Mais Comuns no Mundo — NordPass 2024\n(com tempo estimado de quebra)',
             color='white', fontsize=12, fontweight='bold')
ax.set_xlabel('Usuários Afetados (Milhões)', color='#aaaacc')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}M'))

ax.text(0.98, 0.02,
        'Fonte: NordPass Most Common Passwords 2024',
        transform=ax.transAxes, ha='right', va='bottom',
        color='#888899', fontsize=8, style='italic')

plt.tight_layout()
plt.savefig('outputs/graficos/08_nordpass_top15.png', bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
print('✅ Gráfico 8 salvo em outputs/graficos/08_nordpass_top15.png')

---
## 💡 CÉLULA 11 — Insights e Conclusões da EDA


In [ ]:
# ============================================================
# INSIGHTS E CONCLUSÕES DA ANÁLISE EXPLORATÓRIA
# ============================================================

total_senhas  = len(df_passwords)
pct_fracas    = (df_passwords['forca_label'] == 'Fraca').sum() / total_senhas * 100
pct_instant   = (df_passwords['tempo_quebra'] == 'Instantâneo').sum() / total_senhas * 100
comp_medio    = df_passwords['comprimento'].mean()
pct_s_simbolo = (df_passwords['tem_simbolo'] == 0).sum() / total_senhas * 100
total_afetados = df_breaches['registros_afetados'].sum() if 'registros_afetados' in df_breaches.columns else 0
senha_mais_comum = df_rockyou.iloc[0]['senha']
freq_mais_comum  = df_rockyou.iloc[0]['frequencia']

print('=' * 65)
print('  💡 PRINCIPAIS INSIGHTS DA ANÁLISE EXPLORATÓRIA')
print('=' * 65)

insights = [
    f'🔴 {pct_fracas:.1f}% das senhas analisadas são classificadas como FRACAS',
    f'⚡ {pct_instant:.1f}% das senhas seriam quebradas INSTANTANEAMENTE por força bruta',
    f'📏 O comprimento médio das senhas é de apenas {comp_medio:.1f} caracteres',
    f'🔤 {pct_s_simbolo:.1f}% das senhas NÃO utilizam nenhum símbolo especial',
    f'🏆 A senha mais comum no RockYou é "{senha_mais_comum}" ({freq_mais_comum:,} ocorrências)',
    f'🌍 O vazamento do RockYou (2009) expôs mais de 32 milhões de senhas em texto puro',
    f'📅 Vazamentos de dados aumentaram significativamente a partir de 2012',
    f'🏢 Empresas de Tecnologia e Redes Sociais são os alvos mais frequentes',
    f'⏱ Todas as 15 senhas mais comuns do mundo são quebradas em menos de 1 segundo',
    f'🛡️ Senhas com 4 tipos de caracteres + mais de 12 dígitos levam Décadas para serem quebradas'
]

for i, insight in enumerate(insights, 1):
    print(f'\n  {i:02d}. {insight}')

print('\n' + '=' * 65)
print('  ✅ EDA concluída! 8 gráficos salvos em outputs/graficos/')
print('  → Próximo passo: Notebook 04 — Dashboard e Visualizações Avançadas')
print('=' * 65)